# Responses API overview (Python SDK)

This notebook introduces the Responses API for stateful, tool-using applications. It replaces the deprecated Assistants API; migrate before the Assistants API shutdown on August 26, 2026.


## Prerequisites

Set `OPENAI_API_KEY` in your environment. The API calls below create billable resources. Run only the sections you need, and delete uploaded files and vector stores when you are done.


In [ ]:
%pip install --upgrade openai

import json
import os

from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4o"
INSTRUCTIONS = (
    "You are a personal math tutor. Answer questions briefly, in a sentence or less."
)


## Create a response

Instructions are sent with each request. The response contains both convenient text output and structured output items.


In [ ]:
response = client.responses.create(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input="I need to solve the equation 3x + 11 = 14. Can you help me?",
)

print(response.output_text)


## Continue a conversation

Use `previous_response_id` to continue from a prior response. Keep passing the instructions because they are not automatically carried forward.


In [ ]:
follow_up = client.responses.create(
    model=MODEL,
    instructions=INSTRUCTIONS,
    previous_response_id=response.id,
    input="Could you explain that step in a little more detail?",
)

print(follow_up.output_text)


## Add a custom function

When the model returns a `function_call` output item, execute the function in your application and return a `function_call_output` item in a follow-up response.


In [ ]:
def get_weather(location: str) -> dict:
    # Replace this deterministic example with an API call in your application.
    return {"location": location, "temperature_c": 22, "conditions": "sunny"}

weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Get the current weather for a location.",
    "parameters": {
        "type": "object",
        "properties": {"location": {"type": "string"}},
        "required": ["location"],
        "additionalProperties": False,
    },
    "strict": True,
}

weather_response = client.responses.create(
    model=MODEL,
    input="What is the weather in Phoenix?",
    tools=[weather_tool],
)

tool_call = next(item for item in weather_response.output if item.type == "function_call")
tool_output = get_weather(**json.loads(tool_call.arguments))

weather_answer = client.responses.create(
    model=MODEL,
    previous_response_id=weather_response.id,
    input=[{
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(tool_output),
    }],
    tools=[weather_tool],
)

print(weather_answer.output_text)


## Search uploaded files

File search uses a vector store that you attach directly to the Responses tool. This section uploads a local PDF and then waits for it to finish indexing.


In [ ]:
with open("data/language_models_are_unsupervised_multitask_learners.pdf", "rb") as document:
    uploaded_file = client.files.create(file=document, purpose="user_data")

vector_store = client.vector_stores.create(name="language-models-paper")
client.vector_stores.files.create_and_poll(
    vector_store_id=vector_store.id,
    file_id=uploaded_file.id,
)

file_search_response = client.responses.create(
    model=MODEL,
    input="What does this paper say about few-shot learning?",
    tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
)

print(file_search_response.output_text)


## Use Code Interpreter

Code Interpreter runs in a container. Make uploaded file IDs available through an automatically created container, then inspect the structured output for logs and generated images.


In [ ]:
analysis_response = client.responses.create(
    model=MODEL,
    input="Use Python to summarize the uploaded paper in a small table.",
    tools=[{
        "type": "code_interpreter",
        "container": {"type": "auto", "file_ids": [uploaded_file.id]},
    }],
)

print(analysis_response.output_text)
code_calls = [item for item in analysis_response.output if item.type == "code_interpreter_call"]
for call in code_calls:
    for output in call.outputs or []:
        if output.type == "logs":
            print(output.logs)


## Clean up

Delete resources you no longer need. Deleting a vector store does not delete the uploaded file, so delete both explicitly.


In [ ]:
client.vector_stores.delete(vector_store.id)
client.files.delete(uploaded_file.id)
